In [ ]:
import sys
from pathlib import Path

# 1. Определяем корень проекта
# (подбираем количество .parent, чтобы попасть в max_projects)
project_root = Path.cwd().parent.parent.parent

# 2. Добавляем корень в пути поиска модулей
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# 3. Проверяем, что путь добавлен
print(f"✅ Project root: {project_root}")
print(f"✅ Sys path: {sys.path[:3]}...")

import pandas as pd
from service import WMSStockService
from types import list, dict

✅ Project root: d:\Pytnon_scripts\start_vector
✅ Sys path: ['d:\\Pytnon_scripts\\start_vector', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\python313.zip', 'C:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python313\\DLLs']...


In [2]:
data = await WMSStockService().fetch_external_stocks()  # ✅ Верно

✅ Страница 1: получено 1548 записей
✅ Страница 2: получено 1548 записей
✅ Страница 3: получено 1548 записей
✅ Страница 4: получено 1548 записей
✅ Страница 5: получено 1548 записей
✅ Страница 6: получено 1548 записей
✅ Страница 7: получено 1548 записей
✅ Страница 8: получено 1548 записей
✅ Страница 9: получено 1548 записей
✅ Страница 10: получено 0 записей


In [19]:
def process_historical_stocks(data: list = None)-> pd.DataFrame:
    if data is None:
        return pd.DataFrame()
    stock_list = []
    for item in data:
        wild = item.get("product_id")
        for transaction in item.get("data"):
            stock_list.append({
                "wild": wild,
                "transaction_date": transaction["transaction_date"],
                "end_of_day_balance": transaction["end_of_day_balance"]
            })

    df = pd.DataFrame(stock_list)
    return df

In [20]:
df = process_historical_stocks(data)

In [21]:
df.head()

,wild,transaction_date,end_of_day_balance
0,metawild_test,2026-03-22,0
1,metawild_test,2026-03-21,0
2,metawild_test,2026-03-20,0
3,metawild_test,2026-03-19,0
4,testwild,2026-03-22,620


In [22]:
from tables_scheme import historical_stocks_fbs_service_table

print("Размер df:", df.shape)
print("Колонки:", df.columns.tolist())
print(df.head(10))

unique_keys = historical_stocks_fbs_service_table.get("key_columns")
print("Ключи:", unique_keys)

duplicates = df[df.duplicated(subset=unique_keys, keep=False)].sort_values(by=unique_keys)
print("Количество дублей по ключу:", len(duplicates))
print(duplicates.head(50))


Размер df: (44892, 3)
Колонки: ['wild', 'transaction_date', 'end_of_day_balance']
            wild transaction_date  end_of_day_balance
0  metawild_test       2026-03-22                   0
1  metawild_test       2026-03-21                   0
2  metawild_test       2026-03-20                   0
3  metawild_test       2026-03-19                   0
4       testwild       2026-03-22                 620
5       testwild       2026-03-21                 620
6       testwild       2026-03-20                 620
7       testwild       2026-03-19                 620
8      testwild2       2026-03-22                 116
9      testwild2       2026-03-21                 116
Ключи: ['transaction_date', 'wild']
Количество дублей по ключу: 0
Empty DataFrame
Columns: [wild, transaction_date, end_of_day_balance]
Index: []


In [ ]:
print("table_name:", table_name)
print("unique_keys runtime:", unique_keys)
print("schema_definition:", scheme_definition)